# Param2Image v3.1 — Google Colab TPU training (v5e-1 / v2-8 / v3-8)

TPU-native training script using `torch_xla`. The model code is unchanged from the CUDA version — only the training loop is different.

**Before running**: Runtime > Change runtime type > **TPU** (NOT GPU). Verify the runtime details show `TPU v5e-1` (or whichever variant Colab assigns).

**First iteration warning**: XLA compiles the computational graph on the first iteration (3–10 min). This is normal. Subsequent iterations are very fast. Do NOT interrupt during the first iteration.

**Drive layout** (same as GPU notebook):
```
My Drive/
└── Train_v3/
    ├── train_p2i_xla.py            ← TPU training script (this notebook uses it)
    ├── train_p2i_pytorch.py        ← GPU script (unused here)
    ├── model_p2i_pytorch.py        ← same model code, used by both
    ├── data_loader_p2i_pytorch.py  ← same loader
    ├── preprocessing_p2i.py        ← same scaler
    ├── generate_p2i_pytorch.py     ← inference (CPU/GPU)
    ├── notebook_train_p2i_colab_tpu.ipynb  ← this file
    └── Abaqus_OUT/  (or Abaqus_OUT.zip)
```

## Step 1 — Install torch_xla and verify TPU
Colab usually pre-installs `torch_xla` on TPU runtimes. If the version mismatches your PyTorch, this cell installs the matching pair.

In [ ]:
import sys, subprocess
# Check torch_xla availability without installing if it already works.
try:
    import torch_xla, torch_xla.core.xla_model as xm
    dev = xm.xla_device()
    print(f'torch_xla {torch_xla.__version__} OK | device = {dev}')
except Exception as e:
    print('torch_xla not ready, installing...')
    # Match torch_xla to the preinstalled torch major.minor.
    import torch
    torch_mm = '.'.join(torch.__version__.split('.')[:2])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f'torch_xla[tpu]~={torch_mm}',
                    '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'],
                   check=False)
    import torch_xla, torch_xla.core.xla_model as xm
    dev = xm.xla_device()
    print(f'Installed torch_xla {torch_xla.__version__} | device = {dev}')

import sys, subprocess
# Check torch_xla is importable. DO NOT acquire the TPU device here — if we
# did, the notebook kernel would hold the TPU lock and the training subprocess
# (cell 7) would fail with "Device or resource busy". Device acquisition is
# the training script's job only.
try:
    import torch_xla
    print(f'torch_xla {torch_xla.__version__} OK (device acquisition deferred to training)')
except ImportError:
    print('torch_xla not present, installing...')
    import torch
    torch_mm = '.'.join(torch.__version__.split('.')[:2])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f'torch_xla[tpu]~={torch_mm}',
                    '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'],
                   check=False)
    import torch_xla
    print(f'Installed torch_xla {torch_xla.__version__} (device acquisition deferred)')

In [ ]:
!pip uninstall -y -q pillow
!pip install --no-cache-dir -q pillow-simd
import PIL
print('PIL version:', PIL.__version__)

## Step 3 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_TRAIN_PATH = '/content/drive/MyDrive/Train_v3'
assert os.path.exists(DRIVE_TRAIN_PATH), f'Train_v3 folder not found at {DRIVE_TRAIN_PATH}'
print('Drive entries:')
for f in sorted(os.listdir(DRIVE_TRAIN_PATH)):
    print(' -', f)

## Step 4 — Copy scripts to local SSD

In [ ]:
import shutil
LOCAL_TRAIN_PATH = '/content/Train_v3'
os.makedirs(LOCAL_TRAIN_PATH, exist_ok=True)
n = 0
for f in os.listdir(DRIVE_TRAIN_PATH):
    src = os.path.join(DRIVE_TRAIN_PATH, f)
    if os.path.isfile(src) and f.endswith(('.py', '.md', '.json')):
        shutil.copy2(src, os.path.join(LOCAL_TRAIN_PATH, f))
        n += 1
print(f'Copied {n} script files to {LOCAL_TRAIN_PATH}')
print('Local files:', sorted(os.listdir(LOCAL_TRAIN_PATH)))

## Step 5 — Stage the dataset (prefer zip)

In [ ]:
import zipfile, time
LOCAL_DATA_PATH = '/content/Abaqus_OUT'
DRIVE_ZIP = os.path.join(DRIVE_TRAIN_PATH, 'Abaqus_OUT.zip')
DRIVE_FOLDER = os.path.join(DRIVE_TRAIN_PATH, 'Abaqus_OUT')
CSV_PATH = os.path.join(LOCAL_DATA_PATH, 'results_master.csv')
if os.path.exists(CSV_PATH):
    print(f'Dataset already staged at {LOCAL_DATA_PATH}, skipping.')
elif os.path.exists(DRIVE_ZIP):
    t0 = time.time(); print(f'Extracting {DRIVE_ZIP} -> /content/  ...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf: zf.extractall('/content/')
    print(f'Extracted in {time.time()-t0:.1f} s')
elif os.path.isdir(DRIVE_FOLDER):
    t0 = time.time(); print(f'Copying {DRIVE_FOLDER} -> {LOCAL_DATA_PATH}')
    shutil.copytree(DRIVE_FOLDER, LOCAL_DATA_PATH)
    print(f'Copied in {time.time()-t0:.1f} s')
else:
    raise FileNotFoundError(f'Need {DRIVE_ZIP} or {DRIVE_FOLDER} on Drive.')
assert os.path.exists(CSV_PATH)
import pandas as pd
print(f'Dataset ready: {len(pd.read_csv(CSV_PATH))} simulations.')

## Step 6 — Configure (TPU-tuned defaults)
TPU has 16 GB HBM with bf16 native, so batch can be larger than on T4. Default = 8.

In [ ]:
os.environ['P2I_DATA_DIR'] = LOCAL_DATA_PATH
os.environ['P2I_CSV_PATH'] = CSV_PATH
# TPU checkpoints live in a SEPARATE folder so they don't clash with GPU ones.
LOCAL_CKPT_DIR = os.path.join(LOCAL_TRAIN_PATH, 'training_checkpoints_position_xla')
DRIVE_CKPT_DIR = os.path.join(DRIVE_TRAIN_PATH, 'training_checkpoints_position_xla')
DRIVE_LOG_DIR = os.path.join(DRIVE_TRAIN_PATH, 'logs_position_xla')
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)
os.environ['P2I_CHECKPOINT_DIR']   = LOCAL_CKPT_DIR
os.environ['P2I_DRIVE_BACKUP_DIR'] = DRIVE_CKPT_DIR
os.environ['P2I_LOG_DIR']          = DRIVE_LOG_DIR
os.environ['P2I_TRAIN_LOG_PATH']   = os.path.join(DRIVE_CKPT_DIR, 'train_console_xla.log')
os.environ['P2I_SAVE_EVERY_STEPS'] = '50'
# Resume from latest if it is finite; automatically fall back to best-position if latest is corrupt.
os.environ['P2I_RESUME_CKPT']       = 'auto'
os.environ['P2I_MAX_R1_PENALTY']   = '50'
os.environ['P2I_GRAD_CLIP_NORM']   = '1.0'

# Architecture (same as GPU recipe)
os.environ['P2I_IMG_SIZE']         = '1024'
os.environ['P2I_INIT_SIZE']        = '16'
os.environ['P2I_INIT_CHANNELS']    = '512'
os.environ['P2I_W_DIM']            = '512'
os.environ['P2I_LATENT_DIM']       = '100'

# TPU-tuned training schedule.
# bf16 + 16 GB HBM lets us push batch higher than on T4.
# If first compilation OOMs, drop to batch=4.
os.environ['P2I_BATCH_SIZE']          = '8'
os.environ['P2I_GRAD_ACCUM']          = '1'
os.environ['P2I_USE_GRAD_CHECKPOINT'] = '0'
os.environ['P2I_NUM_WORKERS']         = '2'
os.environ['P2I_EPOCHS']              = '200'
os.environ['P2I_WARMUP_EPOCHS']       = '5'
os.environ['P2I_VAL_SPLIT']           = '0.10'

# Optimizer (same as GPU)
os.environ['P2I_LR_G']    = '2e-5'
os.environ['P2I_LR_D']    = '1e-5'
os.environ['P2I_LR_PHYS'] = '1e-4'
os.environ['P2I_LR_MIN']  = '1e-6'
os.environ['P2I_WD_G']    = '1e-4'
os.environ['P2I_WD_D']    = '1e-4'
os.environ['P2I_WD_PHYS'] = '1e-3'
os.environ['P2I_BETA1_GAN']  = '0.0'
os.environ['P2I_BETA2_GAN']  = '0.99'
os.environ['P2I_BETA1_PHYS'] = '0.9'
os.environ['P2I_BETA2_PHYS'] = '0.999'
os.environ['P2I_EMA_DECAY']  = '0.999'

# Losses
os.environ['P2I_LAMBDA_L1']   = '10'
os.environ['P2I_LAMBDA_NUM']  = '1.0'
# Directly supervise generated impact location against the ground truth image.
os.environ['P2I_LAMBDA_POS']  = '10.0'
os.environ['P2I_R1_GAMMA']    = '1'
os.environ['P2I_R1_INTERVAL'] = '16'
os.environ['P2I_DIFFAUG_POLICY'] = 'color,cutout'

# PhysicsHead
os.environ['P2I_PHYS_HIDDEN']    = '256'
os.environ['P2I_PHYS_DEPTH']     = '6'
os.environ['P2I_PHYS_NUM_FREQS'] = '16'
os.environ['P2I_PHYS_SIGMA']     = '5.0'
os.environ['P2I_MIXUP_ALPHA']    = '0.2'

# XLA-specific env vars (already defaulted in the script, but set here too)
os.environ['XLA_USE_BF16']     = '1'
os.environ['PT_XLA_DEBUG']     = '0'

print('TPU configuration set (batch=8, bf16 native, separate ckpt folder).')
print(f"Drive persistence: checkpoints -> {DRIVE_CKPT_DIR}; logs -> {DRIVE_LOG_DIR}; save every {os.environ['P2I_SAVE_EVERY_STEPS']} optimizer steps.")
print(f"Resume mode: resume={os.environ['P2I_RESUME_CKPT']}, LR_G={os.environ['P2I_LR_G']}, LR_D={os.environ['P2I_LR_D']}, R1={os.environ['P2I_R1_GAMMA']}")

### Restore prior TPU checkpoint from Drive (auto-resume)

In [ ]:
for fname in ('ckpt_latest.pt', 'ckpt_best.pt', 'ckpt_best_position.pt', 'normalization_stats.json', 'config.json'):
    src = os.path.join(DRIVE_CKPT_DIR, fname); dst = os.path.join(LOCAL_CKPT_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f'Restored {fname} from Drive.')
    elif os.path.exists(src):
        print(f'{fname} already on local SSD.')
    else:
        print(f'{fname} not on Drive yet — will be created on first epoch.')

## Step 7 — Train on TPU
First iteration will take 3–10 minutes (XLA compile). After that, each epoch should be 60–90 s.

**Free-tier TPU 2-hour limit**: at ~75 s/epoch, you can complete ~80 epochs per session. With auto-resume from Drive across sessions, all 200 epochs are reachable in 3 sessions.

In [ ]:
import sys
sys.path.insert(0, LOCAL_TRAIN_PATH); os.chdir(LOCAL_TRAIN_PATH)
!python -u train_p2i_xla.py

## Step 8 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_LOG_DIR}

## Step 9 — Generate a sample
Inference is currently CPU/GPU only via `generate_p2i_pytorch.py`. To use the TPU-trained checkpoint, point `P2I_CHECKPOINT_DIR` at the XLA folder.

In [ ]:
os.environ['P2I_CHECKPOINT_DIR'] = LOCAL_CKPT_DIR  # the XLA checkpoint dir
!python generate_p2i_pytorch.py --velocity 200 --radius 10 --impact_x 0 --impact_y 0 --seed 0

## Step 10 — Mirror logs + generations to Drive

In [ ]:
for sub in ('Prediction_OUT',):
    src_dir = os.path.join(LOCAL_TRAIN_PATH, sub); dst_dir = os.path.join(DRIVE_TRAIN_PATH, sub)
    if os.path.isdir(src_dir):
        os.makedirs(dst_dir, exist_ok=True)
        for root, _, files in os.walk(src_dir):
            rel = os.path.relpath(root, src_dir)
            target = os.path.join(dst_dir, rel) if rel != '.' else dst_dir
            os.makedirs(target, exist_ok=True)
            for f in files:
                try: shutil.copy2(os.path.join(root, f), os.path.join(target, f))
                except Exception as e: print(f'skipped {f}: {e}')
print('Mirror complete.')